In [0]:
"""
sim_invoice_delta_fixed.py
==========================
Fixed & fully-connected invoice simulation for smart_odoo.silver

FIXES vs original
─────────────────
FIX 1  : state = draft(10%) | posted(80%) | cancel(10%) — was always "posted"
FIX 2  : partner_id loaded from silver.res_partner
          out_invoice → customers (customer_rank > 0, id 51-250)
          in_invoice  → vendors   (supplier_rank > 0, id 1-50)
FIX 3  : product names/prices loaded from silver.product_template &
          silver.product_product — no more fake "Product_001" strings
FIX 4  : account_journal, account_tax, account_payment_term use MERGE
          on PK instead of blind append → zero duplicates on re-run
FIX 5  : invoice_origin references real sale_order_id (out_invoice)
          or real purchase_order_id (in_invoice) from silver tables
FIX 6  : invoice_user_id / user_name loaded from silver.res_users
FIX 7  : account_account uses MERGE ON account_id → no duplicates
FIX 8  : account_analytic_account uses MERGE ON analytic_account_id
FIX 9  : price_unit = real product price (list_price for sales,
          standard_price for purchases) — qty drives total, not reverse
FIX 10 : cancel invoices: amount_residual = 0, payment_state = "not_paid"
          (can't have outstanding balance on a cancelled invoice)
FIX 11 : team_id comes from the linked sale_order (out_invoice only)
ADDED  : account_move_line now carries sale_order_line_id (out_invoice)
         or purchase_order_line_id (in_invoice) for cross-module joins
"""

import random
from datetime import datetime, timedelta, timezone

from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType,
    BooleanType, TimestampType,
)

# ─────────────────────────────────────────────────────────────
# SPARK + CATALOG
# ─────────────────────────────────────────────────────────────
spark = SparkSession.builder.appName("sim_invoice_delta_fixed").getOrCreate()

CATALOG    = "smart_odoo"
SCHEMA     = "silver"
SOURCE_DB  = "python_sim"
COMPANY_ID = 1
COMPANY    = "Smart Odoo LLC"
TAX_RATE   = 0.14   # Egypt VAT

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# ─────────────────────────────────────────────────────────────
# DATE HELPERS
# ─────────────────────────────────────────────────────────────
START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 6, 15)
now        = datetime.now(timezone.utc).replace(tzinfo=None)

def rand_date() -> datetime:
    return START_DATE + timedelta(
        days=random.randint(0, (END_DATE - START_DATE).days)
    )

def maybe(val, prob_none: float = 0.3):
    return None if random.random() < prob_none else val

# ─────────────────────────────────────────────────────────────
# DELTA HELPERS
# ─────────────────────────────────────────────────────────────
def max_id(table: str, col: str) -> int:
    try:
        return int(spark.sql(
            f"SELECT COALESCE(MAX({col}), 0) AS m "
            f"FROM {CATALOG}.{SCHEMA}.{table}"
        ).collect()[0]["m"])
    except Exception:
        return 0

def merge_delta(df, table: str, pk: str) -> None:
    """Idempotent MERGE on PK — safe on DROP TABLE + re-run."""
    full_table = f"{CATALOG}.{SCHEMA}.{table}"
    view_name  = f"_inv_src_{table}"
    try:
        spark.catalog.tableExists(full_table)
        exists = spark.sql(
            f"SELECT COUNT(*) AS c FROM {CATALOG}.information_schema.tables "
            f"WHERE table_schema='{SCHEMA}' AND table_name='{table}'"
        ).collect()[0]["c"] > 0
    except Exception:
        exists = False

    if not exists:
        (df.write.format("delta").mode("errorifexists")
           .saveAsTable(full_table))
        print(f"  ✅  {table}  (created, {df.count()} rows)")
        return

    df.createOrReplaceTempView(view_name)
    other_cols = [c for c in df.columns if c not in (pk, "dwh_loaded_at")]
    set_clause = ",\n            ".join(
        [f"t.{c} = s.{c}" for c in other_cols]
        + ["t.dwh_loaded_at = current_timestamp()"]
    )
    spark.sql(f"""
        MERGE INTO {full_table} AS t
        USING {view_name}       AS s
        ON t.{pk} = s.{pk}
        WHEN MATCHED     THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED THEN INSERT *
    """)
    spark.catalog.dropTempView(view_name)
    print(f"  ✅  {table}  (merged, {df.count()} rows)")

def append_delta(df, table: str) -> None:
    (df.write.format("delta").mode("append")
       .option("mergeSchema", "true")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}"))
    print(f"  ✅  {table}  (+{df.count()} rows)")
    
# ─────────────────────────────────────────────────────────────
# FIX 2 — Load real partners from silver.res_partner
# ─────────────────────────────────────────────────────────────
customers_map = {}   # partner_id → partner_name  (customer_rank > 0)
vendors_map   = {}   # partner_id → partner_name  (supplier_rank > 0)

for r in spark.sql("""
    SELECT partner_id, partner_name, customer_rank, supplier_rank
    FROM   silver.res_partner
""").collect():
    if r.customer_rank and r.customer_rank > 0:
        customers_map[r.partner_id] = r.partner_name
    if r.supplier_rank and r.supplier_rank > 0:
        vendors_map[r.partner_id] = r.partner_name

customer_list = sorted(customers_map.keys())
vendor_list   = sorted(vendors_map.keys())

assert customer_list, "❌ No customers found in silver.res_partner"
assert vendor_list,   "❌ No vendors found in silver.res_partner"
print(f"  Partners loaded  : {len(vendor_list)} vendors | {len(customer_list)} customers")

# ─────────────────────────────────────────────────────────────
# FIX 3 — Load real products from silver
# ─────────────────────────────────────────────────────────────
products_map = {}   # product_id → {name, list_price, standard_price}

for r in spark.sql("""
    SELECT t.product_tmpl_id AS pid,
           t.name,
           t.list_price,
           p.standard_price
    FROM   silver.product_template t
    JOIN   silver.product_product  p ON p.product_id = t.product_tmpl_id
""").collect():
    products_map[r.pid] = {
        "name":           r.name,
        "list_price":     float(r.list_price),
        "standard_price": float(r.standard_price),
    }

pid_list = sorted(products_map.keys())
assert pid_list, "❌ No products found in silver.product_template"
print(f"  Products loaded  : {len(pid_list)}")

# ─────────────────────────────────────────────────────────────
# FIX 5 — Load real sale_order and purchase_order IDs
# ─────────────────────────────────────────────────────────────
# out_invoice links to sale_order (confirmed/done orders)
sale_orders = {}   # sale_order_id → {partner_id, team_id, team_name, line_ids[]}
try:
    for r in spark.sql("""
        SELECT sale_order_id, partner_id, team_id, team_name
        FROM   silver.sale_order
        WHERE  order_state IN ('sale', 'done')
        LIMIT  1000
    """).collect():
        sale_orders[r.sale_order_id] = {
            "partner_id": r.partner_id,
            "team_id":    r.team_id,
            "team_name":  r.team_name,
            "line_ids":   [],
        }
    # Load line IDs per order
    for r in spark.sql("""
        SELECT sale_order_id, sale_order_line_id
        FROM   silver.sale_order_line
        WHERE  order_state IN ('sale', 'done')
        LIMIT  5000
    """).collect():
        if r.sale_order_id in sale_orders:
            sale_orders[r.sale_order_id]["line_ids"].append(r.sale_order_line_id)
except Exception as e:
    print(f"  ⚠  sale_order not available: {e}")

so_ids = sorted(sale_orders.keys())

# in_invoice links to purchase_order (confirmed/done orders)
purchase_orders = {}   # po_id → {partner_id, line_ids[]}
try:
    for r in spark.sql("""
        SELECT purchase_order_id, partner_id
        FROM   silver.purchase_order
        WHERE  order_state IN ('purchase', 'done')
        LIMIT  1000
    """).collect():
        purchase_orders[r.purchase_order_id] = {
            "partner_id": r.partner_id,
            "line_ids":   [],
        }
    for r in spark.sql("""
        SELECT order_id, purchase_order_line_id
        FROM   silver.purchase_order_line
        LIMIT  5000
    """).collect():
        if r.order_id in purchase_orders:
            purchase_orders[r.order_id]["line_ids"].append(r.purchase_order_line_id)
except Exception as e:
    print(f"  ⚠  purchase_order not available: {e}")

po_ids = sorted(purchase_orders.keys())
print(f"  Sale orders      : {len(so_ids)} | Purchase orders: {len(po_ids)}")

# ─────────────────────────────────────────────────────────────
# FIX 6 — Load real users from silver.res_users
# ─────────────────────────────────────────────────────────────
users_map = {}   # user_id → partner_name (used as display name)
try:
    for r in spark.sql("""
        SELECT user_id, partner_name
        FROM   silver.res_users
    """).collect():
        users_map[r.user_id] = r.partner_name
except Exception:
    users_map = {i: f"User_{i}" for i in range(1, 11)}

user_list = sorted(users_map.keys())
print(f"  Users loaded     : {len(user_list)}")

# ─────────────────────────────────────────────────────────────
# STATIC REFERENCE DATA
# ─────────────────────────────────────────────────────────────
CURRENCIES = {1: ("EGP", 1.0), 2: ("USD", 48.5), 3: ("EUR", 52.0)}

JOURNAL_SEED = [
    (1, "SAL",  "Sales Journal",    "sale"),
    (2, "PUR",  "Purchase Journal", "purchase"),
    (3, "BNK",  "Bank Journal",     "bank"),
    (4, "CSH",  "Cash Journal",     "cash"),
    (5, "MISC", "General Journal",  "general"),
]
JOURNALS      = {r[0]: (r[1], r[2], r[3]) for r in JOURNAL_SEED}
SALE_JIDS     = [1]
PURCHASE_JIDS = [2]
BANK_JIDS     = [3, 4]

TAX_SEED = [
    (1, "VAT 14%",  14.0, "sale",     "T1", "S"),
    (2, "VAT 5%",    5.0, "purchase", "T2", "S"),
    (3, "Exempt",    0.0, "sale",     "T3", "E"),
]
TAXES = {r[0]: (r[1], r[2]) for r in TAX_SEED}

PT_SEED = [
    (1, "Immediate Payment", 1,  0,  0.0,  False),
    (2, "Net 30",            2, 30,  0.0,  False),
    (3, "Net 60",            3, 60,  0.0,  False),
    (4, "2/10 Net 30",       4, 30,  2.0,  True),
]
PAYMENT_TERMS = {r[0]: r[1] for r in PT_SEED}

ACC_TYPES = [
    "asset_current", "asset_receivable", "asset_cash", "asset_fixed",
    "liability_current", "liability_payable", "equity",
    "income", "expense", "expense_direct_cost",
]
NAMES_MAP = {
    "asset_current":        ["Current Assets", "Short Term Assets"],
    "asset_receivable":     ["Accounts Receivable", "Customer Receivables"],
    "asset_cash":           ["Cash", "Bank"],
    "asset_fixed":          ["Fixed Assets", "Equipment"],
    "liability_current":    ["Current Liabilities"],
    "liability_payable":    ["Accounts Payable"],
    "equity":               ["Capital", "Owner Equity"],
    "income":               ["Sales Revenue", "Operating Income"],
    "expense":              ["Operating Expenses"],
    "expense_direct_cost":  ["COGS"],
}
CODE_PREFIX = {
    "asset_current": "101", "asset_receivable": "121", "asset_cash": "1014",
    "asset_fixed": "151", "liability_current": "201", "liability_payable": "211",
    "equity": "301", "income": "400", "expense": "600", "expense_direct_cost": "500",
}
UOMS  = {1: "Unit(s)", 2: "kg", 3: "liters", 4: "pcs", 5: "boxes"}
PLANS = {1: "Cost Center", 2: "Project", 3: "Department"}
PAYMENT_METHODS = {1: "Manual", 2: "Bank Transfer", 3: "Check"}
PM_POOL = [1]*60 + [2]*30 + [3]*10

# ─────────────────────────────────────────────────────────────
# SCHEMAS
# ─────────────────────────────────────────────────────────────
j_schema = StructType([
    StructField("account_journal_id",        IntegerType(),   True),
    StructField("company_id",                IntegerType(),   True),
    StructField("company_name",              StringType(),    True),
    StructField("currency_id",               IntegerType(),   True),
    StructField("currency_name",             StringType(),    True),
    StructField("default_account_id",        IntegerType(),   True),
    StructField("default_account_name",      StringType(),    True),
    StructField("sequence",                  IntegerType(),   True),
    StructField("code",                      StringType(),    True),
    StructField("type",                      StringType(),    True),
    StructField("name",                      StringType(),    True),
    StructField("bank_statements_source",    StringType(),    True),
    StructField("invoice_reference_type",    StringType(),    True),
    StructField("invoice_reference_model",   StringType(),    True),
    StructField("active",                    BooleanType(),   True),
    StructField("is_self_billing",           BooleanType(),   True),
    StructField("restrict_mode_hash_table",  BooleanType(),   True),
    StructField("refund_sequence",           BooleanType(),   True),
    StructField("payment_sequence",          BooleanType(),   True),
    StructField("show_on_dashboard",         BooleanType(),   True),
    StructField("created_at",                TimestampType(), True),
    StructField("updated_at",                TimestampType(), True),
    StructField("dwh_loaded_at",             TimestampType(), True),
    StructField("dwh_source_db",             StringType(),    True),
])

tax_schema = StructType([
    StructField("account_tax_id",            IntegerType(),   True),
    StructField("company_id",                IntegerType(),   True),
    StructField("company_name",              StringType(),    True),
    StructField("tax_group_id",              IntegerType(),   True),
    StructField("tax_group_name",            StringType(),    True),
    StructField("country_id",                IntegerType(),   True),
    StructField("country_name",              StringType(),    True),
    StructField("sequence",                  IntegerType(),   True),
    StructField("type_tax_use",              StringType(),    True),
    StructField("tax_scope",                 StringType(),    True),
    StructField("amount_type",               StringType(),    True),
    StructField("tax_exigibility",           StringType(),    True),
    StructField("name",                      StringType(),    True),
    StructField("description",               StringType(),    True),
    StructField("invoice_label",             StringType(),    True),
    StructField("ubl_cii_tax_category_code", StringType(),    True),
    StructField("l10n_eg_eta_code",          StringType(),    True),
    StructField("amount",                    DoubleType(),    True),
    StructField("is_domestic",               BooleanType(),   True),
    StructField("active",                    BooleanType(),   True),
    StructField("include_base_amount",       BooleanType(),   True),
    StructField("is_base_affected",          BooleanType(),   True),
    StructField("analytic",                  BooleanType(),   True),
    StructField("created_at",                TimestampType(), True),
    StructField("updated_at",                TimestampType(), True),
    StructField("dwh_loaded_at",             TimestampType(), True),
    StructField("dwh_source_db",             StringType(),    True),
])

pt_schema = StructType([
    StructField("account_payment_term_id",         IntegerType(),   True),
    StructField("company_id",                      IntegerType(),   True),
    StructField("company_name",                    StringType(),    True),
    StructField("sequence",                        IntegerType(),   True),
    StructField("discount_days",                   IntegerType(),   True),
    StructField("early_pay_discount_computation",  StringType(),    True),
    StructField("name",                            StringType(),    True),
    StructField("note",                            StringType(),    True),
    StructField("discount_percentage",             DoubleType(),    True),
    StructField("active",                          BooleanType(),   True),
    StructField("display_on_invoice",              BooleanType(),   True),
    StructField("early_discount",                  BooleanType(),   True),
    StructField("created_at",                      TimestampType(), True),
    StructField("updated_at",                      TimestampType(), True),
    StructField("dwh_loaded_at",                   TimestampType(), True),
    StructField("dwh_source_db",                   StringType(),    True),
])

acc_schema = StructType([
    StructField("account_id",    IntegerType(),   True),
    StructField("currency_id",   IntegerType(),   True),
    StructField("currency_name", StringType(),    True),
    StructField("account_type",  StringType(),    True),
    StructField("name",          StringType(),    True),
    StructField("code_store",    StringType(),    True),
    StructField("note",          StringType(),    True),
    StructField("active",        BooleanType(),   True),
    StructField("reconcile",     BooleanType(),   True),
    StructField("non_trade",     BooleanType(),   True),
    StructField("created_at",    TimestampType(), True),
    StructField("updated_at",    TimestampType(), True),
    StructField("dwh_loaded_at", TimestampType(), True),
    StructField("dwh_source_db", StringType(),    True),
])

ana_schema = StructType([
    StructField("analytic_account_id", IntegerType(),   True),
    StructField("plan_id",             IntegerType(),   True),
    StructField("plan_name",           StringType(),    True),
    StructField("company_id",          IntegerType(),   True),
    StructField("company_name",        StringType(),    True),
    StructField("partner_id",          IntegerType(),   True),
    StructField("partner_name",        StringType(),    True),
    StructField("code",                StringType(),    True),
    StructField("name",                StringType(),    True),
    StructField("active",              BooleanType(),   True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("dwh_loaded_at",       TimestampType(), True),
    StructField("dwh_source_db",       StringType(),    True),
])

bs_schema = StructType([
    StructField("bank_statement_id", IntegerType(),   True),
    StructField("company_id",        IntegerType(),   True),
    StructField("company_name",      StringType(),    True),
    StructField("currency_id",       IntegerType(),   True),
    StructField("currency_name",     StringType(),    True),
    StructField("journal_id",        IntegerType(),   True),
    StructField("journal_name",      StringType(),    True),
    StructField("name",              StringType(),    True),
    StructField("reference",         StringType(),    True),
    StructField("balance_start",     DoubleType(),    True),
    StructField("balance_end",       DoubleType(),    True),
    StructField("balance_end_real",  DoubleType(),    True),
    StructField("is_complete",       BooleanType(),   True),
    StructField("date",              TimestampType(), True),
    StructField("created_at",        TimestampType(), True),
    StructField("updated_at",        TimestampType(), True),
    StructField("dwh_loaded_at",     TimestampType(), True),
    StructField("dwh_source_db",     StringType(),    True),
])

bsl_schema = StructType([
    StructField("bank_statement_line_id", IntegerType(),   True),
    StructField("statement_id",           IntegerType(),   True),
    StructField("move_id",                IntegerType(),   True),
    StructField("move_name",              StringType(),    True),
    StructField("journal_id",             IntegerType(),   True),
    StructField("journal_name",           StringType(),    True),
    StructField("company_id",             IntegerType(),   True),
    StructField("company_name",           StringType(),    True),
    StructField("partner_id",             IntegerType(),   True),
    StructField("partner_name",           StringType(),    True),
    StructField("currency_id",            IntegerType(),   True),
    StructField("currency_name",          StringType(),    True),
    StructField("foreign_currency_id",    IntegerType(),   True),
    StructField("foreign_currency_name",  StringType(),    True),
    StructField("sequence",               IntegerType(),   True),
    StructField("account_number",         StringType(),    True),
    StructField("transaction_type",       StringType(),    True),
    StructField("payment_ref",            StringType(),    True),
    StructField("amount",                 DoubleType(),    True),
    StructField("amount_currency",        DoubleType(),    True),
    StructField("amount_residual",        DoubleType(),    True),
    StructField("is_reconciled",          BooleanType(),   True),
    StructField("created_at",             TimestampType(), True),
    StructField("updated_at",             TimestampType(), True),
    StructField("dwh_loaded_at",          TimestampType(), True),
    StructField("dwh_source_db",          StringType(),    True),
])

move_schema = StructType([
    StructField("account_move_id",               IntegerType(),   True),
    StructField("journal_id",                    IntegerType(),   True),
    StructField("journal_name",                  StringType(),    True),
    StructField("company_id",                    IntegerType(),   True),
    StructField("company_name",                  StringType(),    True),
    StructField("partner_id",                    IntegerType(),   True),
    StructField("partner_name",                  StringType(),    True),
    StructField("partner_shipping_id",           IntegerType(),   True),
    StructField("partner_shipping_name",         StringType(),    True),
    StructField("currency_id",                   IntegerType(),   True),
    StructField("currency_name",                 StringType(),    True),
    StructField("invoice_payment_term_id",       IntegerType(),   True),
    StructField("invoice_payment_term_name",     StringType(),    True),
    StructField("fiscal_position_id",            IntegerType(),   True),
    StructField("fiscal_position_name",          StringType(),    True),
    StructField("invoice_user_id",               IntegerType(),   True),
    StructField("invoice_user_name",             StringType(),    True),
    StructField("reversed_entry_id",             IntegerType(),   True),
    StructField("campaign_id",                   IntegerType(),   True),
    StructField("campaign_name",                 StringType(),    True),
    StructField("team_id",                       IntegerType(),   True),
    StructField("team_name",                     StringType(),    True),
    # source order FKs
    StructField("sale_order_id",                 IntegerType(),   True),
    StructField("purchase_order_id",             IntegerType(),   True),
    StructField("sequence_prefix",               StringType(),    True),
    StructField("name",                          StringType(),    True),
    StructField("ref",                           StringType(),    True),
    StructField("state",                         StringType(),    True),  # FIX 1
    StructField("move_type",                     StringType(),    True),
    StructField("auto_post",                     StringType(),    True),
    StructField("payment_reference",             StringType(),    True),
    StructField("payment_state",                 StringType(),    True),
    StructField("invoice_origin",                StringType(),    True),  # FIX 5
    StructField("invoice_partner_display_name",  StringType(),    True),
    StructField("narration",                     StringType(),    True),
    StructField("invoice_currency_rate",         DoubleType(),    True),
    StructField("amount_untaxed",                DoubleType(),    True),
    StructField("amount_tax",                    DoubleType(),    True),
    StructField("amount_total",                  DoubleType(),    True),
    StructField("amount_residual",               DoubleType(),    True),
    StructField("amount_untaxed_signed",         DoubleType(),    True),
    StructField("amount_tax_signed",             DoubleType(),    True),
    StructField("amount_total_signed",           DoubleType(),    True),
    StructField("amount_residual_signed",        DoubleType(),    True),
    StructField("always_tax_exigible",           BooleanType(),   True),
    StructField("checked",                       BooleanType(),   True),
    StructField("posted_before",                 BooleanType(),   True),
    StructField("is_move_sent",                  BooleanType(),   True),
    StructField("date",                          TimestampType(), True),
    StructField("invoice_date",                  TimestampType(), True),
    StructField("invoice_date_due",              TimestampType(), True),
    StructField("delivery_date",                 TimestampType(), True),
    StructField("auto_post_until",               TimestampType(), True),
    StructField("created_at",                    TimestampType(), True),
    StructField("updated_at",                    TimestampType(), True),
    StructField("dwh_loaded_at",                 TimestampType(), True),
    StructField("dwh_source_db",                 StringType(),    True),
])

ml_schema = StructType([
    StructField("account_move_line_id",    IntegerType(),   True),
    StructField("move_id",                 IntegerType(),   True),
    StructField("account_id",              IntegerType(),   True),
    StructField("account_name",            StringType(),    True),
    StructField("journal_id",              IntegerType(),   True),
    StructField("journal_name",            StringType(),    True),
    StructField("company_id",              IntegerType(),   True),
    StructField("company_name",            StringType(),    True),
    StructField("currency_id",             IntegerType(),   True),
    StructField("currency_name",           StringType(),    True),
    StructField("partner_id",              IntegerType(),   True),
    StructField("partner_name",            StringType(),    True),
    StructField("product_id",              IntegerType(),   True),
    StructField("product_name",            StringType(),    True),
    StructField("product_uom_id",          IntegerType(),   True),
    StructField("product_uom_name",        StringType(),    True),
    StructField("tax_line_id",             IntegerType(),   True),
    StructField("tax_line_name",           StringType(),    True),
    StructField("tax_group_id",            IntegerType(),   True),
    StructField("tax_group_name",          StringType(),    True),
    StructField("payment_id",              IntegerType(),   True),
    # ADDED: source order line FKs
    StructField("sale_order_line_id",      IntegerType(),   True),
    StructField("purchase_order_line_id",  IntegerType(),   True),
    StructField("sequence",                IntegerType(),   True),
    StructField("move_name",               StringType(),    True),
    StructField("parent_state",            StringType(),    True),
    StructField("ref",                     StringType(),    True),
    StructField("name",                    StringType(),    True),
    StructField("display_type",            StringType(),    True),
    StructField("analytic_distribution",   StringType(),    True),
    StructField("debit",                   DoubleType(),    True),
    StructField("credit",                  DoubleType(),    True),
    StructField("balance",                 DoubleType(),    True),
    StructField("amount_currency",         DoubleType(),    True),
    StructField("tax_base_amount",         DoubleType(),    True),
    StructField("amount_residual",         DoubleType(),    True),
    StructField("quantity",                DoubleType(),    True),
    StructField("price_unit",              DoubleType(),    True),
    StructField("price_subtotal",          DoubleType(),    True),
    StructField("price_total",             DoubleType(),    True),
    StructField("discount",                DoubleType(),    True),
    StructField("is_storno",               BooleanType(),   True),
    StructField("reconciled",              BooleanType(),   True),
    StructField("is_downpayment",          BooleanType(),   True),
    StructField("date",                    TimestampType(), True),
    StructField("invoice_date",            TimestampType(), True),
    StructField("date_maturity",           TimestampType(), True),
    StructField("created_at",              TimestampType(), True),
    StructField("updated_at",              TimestampType(), True),
    StructField("dwh_loaded_at",           TimestampType(), True),
    StructField("dwh_source_db",           StringType(),    True),
])

pay_schema = StructType([
    StructField("account_payment_id",             IntegerType(),   True),
    StructField("move_id",                        IntegerType(),   True),
    StructField("move_name",                      StringType(),    True),
    StructField("journal_id",                     IntegerType(),   True),
    StructField("journal_name",                   StringType(),    True),
    StructField("company_id",                     IntegerType(),   True),
    StructField("company_name",                   StringType(),    True),
    StructField("partner_id",                     IntegerType(),   True),
    StructField("partner_name",                   StringType(),    True),
    StructField("currency_id",                    IntegerType(),   True),
    StructField("currency_name",                  StringType(),    True),
    StructField("source_payment_id",              IntegerType(),   True),
    StructField("payment_method_id",              IntegerType(),   True),
    StructField("payment_method_name",            StringType(),    True),
    StructField("name",                           StringType(),    True),
    StructField("state",                          StringType(),    True),
    StructField("payment_type",                   StringType(),    True),
    StructField("partner_type",                   StringType(),    True),
    StructField("memo",                           StringType(),    True),
    StructField("payment_reference",              StringType(),    True),
    StructField("amount",                         DoubleType(),    True),
    StructField("amount_company_currency_signed", DoubleType(),    True),
    StructField("is_reconciled",                  BooleanType(),   True),
    StructField("is_matched",                     BooleanType(),   True),
    StructField("is_sent",                        BooleanType(),   True),
    StructField("date",                           TimestampType(), True),
    StructField("created_at",                     TimestampType(), True),
    StructField("updated_at",                     TimestampType(), True),
    StructField("dwh_loaded_at",                  TimestampType(), True),
    StructField("dwh_source_db",                  StringType(),    True),
])

anal_schema = StructType([
    StructField("analytic_line_id",      IntegerType(),   True),
    StructField("account_id",            IntegerType(),   True),
    StructField("account_name",          StringType(),    True),
    StructField("product_id",            IntegerType(),   True),
    StructField("product_name",          StringType(),    True),
    StructField("product_uom_id",        IntegerType(),   True),
    StructField("product_uom_name",      StringType(),    True),
    StructField("partner_id",            IntegerType(),   True),
    StructField("partner_name",          StringType(),    True),
    StructField("company_id",            IntegerType(),   True),
    StructField("company_name",          StringType(),    True),
    StructField("currency_id",           IntegerType(),   True),
    StructField("currency_name",         StringType(),    True),
    StructField("journal_id",            IntegerType(),   True),
    StructField("journal_name",          StringType(),    True),
    StructField("general_account_id",    IntegerType(),   True),
    StructField("general_account_name",  StringType(),    True),
    StructField("name",                  StringType(),    True),
    StructField("category",              StringType(),    True),
    StructField("code",                  StringType(),    True),
    StructField("ref",                   StringType(),    True),
    StructField("amount",                DoubleType(),    True),
    StructField("unit_amount",           DoubleType(),    True),
    StructField("date",                  TimestampType(), True),
    StructField("created_at",            TimestampType(), True),
    StructField("updated_at",            TimestampType(), True),
    StructField("dwh_loaded_at",         TimestampType(), True),
    StructField("dwh_source_db",         StringType(),    True),
])

# ═══════════════════════════════════════════════════════════════
# 1. SEEDED REFERENCE TABLES  (MERGE ON PK — FIX 4)
# ═══════════════════════════════════════════════════════════════

# account_journal
j_rows = [
    Row(
        account_journal_id        = jid,
        company_id                = COMPANY_ID,
        company_name              = COMPANY,
        currency_id               = 1,
        currency_name             = "EGP",
        default_account_id        = None,
        default_account_name      = None,
        sequence                  = jid,
        code                      = code,
        type                      = jtype,
        name                      = jname,
        bank_statements_source    = "file_import" if jtype in ("bank","cash") else "undefined",
        invoice_reference_type    = "sequence",
        invoice_reference_model   = "odoo",
        active                    = True,
        is_self_billing           = False,
        restrict_mode_hash_table  = False,
        refund_sequence           = True,
        payment_sequence          = True,
        show_on_dashboard         = jtype in ("bank", "cash"),
        created_at                = now,
        updated_at                = now,
        dwh_loaded_at             = now,
        dwh_source_db             = SOURCE_DB,
    )
    for jid, code, jname, jtype in JOURNAL_SEED
]
merge_delta(spark.createDataFrame(j_rows, schema=j_schema),
            "account_journal", "account_journal_id")

# account_tax
tax_rows = [
    Row(
        account_tax_id           = tid,
        company_id               = COMPANY_ID,
        company_name             = COMPANY,
        tax_group_id             = 1,
        tax_group_name           = "Tax Group 1",
        country_id               = 1,
        country_name             = "Egypt",
        sequence                 = tid,
        type_tax_use             = use,
        tax_scope                = None,
        amount_type              = "percent",
        tax_exigibility          = "on_invoice",
        name                     = tname,
        description              = tname,
        invoice_label            = tname,
        ubl_cii_tax_category_code= ubl,
        l10n_eg_eta_code         = eta,
        amount                   = amount,
        is_domestic              = True,
        active                   = True,
        include_base_amount      = False,
        is_base_affected         = True,
        analytic                 = False,
        created_at               = now,
        updated_at               = now,
        dwh_loaded_at            = now,
        dwh_source_db            = SOURCE_DB,
    )
    for tid, tname, amount, use, eta, ubl in TAX_SEED
]
merge_delta(spark.createDataFrame(tax_rows, schema=tax_schema),
            "account_tax", "account_tax_id")

# account_payment_term
pt_rows = [
    Row(
        account_payment_term_id         = ptid,
        company_id                      = COMPANY_ID,
        company_name                    = COMPANY,
        sequence                        = seq,
        discount_days                   = ddays,
        early_pay_discount_computation  = "mixed" if edisc else None,
        name                            = ptname,
        note                            = None,
        discount_percentage             = dpct,
        active                          = True,
        display_on_invoice              = True,
        early_discount                  = edisc,
        created_at                      = now,
        updated_at                      = now,
        dwh_loaded_at                   = now,
        dwh_source_db                   = SOURCE_DB,
    )
    for ptid, ptname, seq, ddays, dpct, edisc in PT_SEED
]
merge_delta(spark.createDataFrame(pt_rows, schema=pt_schema),
            "account_payment_term", "account_payment_term_id")

# ═══════════════════════════════════════════════════════════════
# 2. ACCOUNT_ACCOUNT — 120 new rows per run, MERGE ON PK (FIX 7)
# ═══════════════════════════════════════════════════════════════
acc_id_start = max_id("account_account", "account_id") + 1

acc_rows = []
acc_id   = acc_id_start
acc_pool: dict[str, list[int]] = {t: [] for t in ACC_TYPES}

for _ in range(120):
    atype     = random.choice(ACC_TYPES)
    aname     = random.choice(NAMES_MAP[atype])
    code      = f"{CODE_PREFIX[atype]}{random.randint(100, 999)}"
    cur_id    = None if random.random() < 0.7 else random.randint(1, 3)
    cur_name  = CURRENCIES[cur_id][0] if cur_id else None
    reconcile = atype in ("asset_receivable", "liability_payable")
    non_trade = atype == "equity"
    cdate     = rand_date()

    acc_rows.append(Row(
        account_id    = acc_id,
        currency_id   = cur_id,
        currency_name = cur_name,
        account_type  = atype,
        name          = aname,
        code_store    = code,
        note          = f"{aname} account",
        active        = True,
        reconcile     = reconcile,
        non_trade     = non_trade,
        created_at    = cdate,
        updated_at    = cdate,
        dwh_loaded_at = now,
        dwh_source_db = SOURCE_DB,
    ))
    acc_pool[atype].append(acc_id)
    acc_id += 1

# Also load existing accounts for FK pool
try:
    for r in spark.sql(
        f"SELECT account_id, account_type FROM {CATALOG}.{SCHEMA}.account_account"
    ).collect():
        if r.account_type in acc_pool:
            acc_pool[r.account_type].append(r.account_id)
except Exception:
    pass

merge_delta(spark.createDataFrame(acc_rows, schema=acc_schema),
            "account_account", "account_id")

def pick_acc(atype: str) -> int:
    pool = acc_pool.get(atype, [])
    return random.choice(pool) if pool else acc_id_start

# ═══════════════════════════════════════════════════════════════
# 3. ACCOUNT_ANALYTIC_ACCOUNT — 30 new per run, MERGE (FIX 8)
# ═══════════════════════════════════════════════════════════════
ana_id_start = max_id("account_analytic_account", "analytic_account_id") + 1

ana_rows = []
ana_id   = ana_id_start
ana_pool = []

for _ in range(30):
    plan_id = random.choice(list(PLANS.keys()))
    pid     = maybe(random.choice(vendor_list + customer_list), 0.4)
    pname   = (vendors_map.get(pid) or customers_map.get(pid)) if pid else None
    ana_rows.append(Row(
        analytic_account_id = ana_id,
        plan_id             = plan_id,
        plan_name           = PLANS[plan_id],
        company_id          = COMPANY_ID,
        company_name        = COMPANY,
        partner_id          = pid,
        partner_name        = pname,
        code                = f"ANA-{ana_id:05d}",
        name                = f"{PLANS[plan_id]} {ana_id}",
        active              = True,
        created_at          = now,
        updated_at          = now,
        dwh_loaded_at       = now,
        dwh_source_db       = SOURCE_DB,
    ))
    ana_pool.append(ana_id)
    ana_id += 1

try:
    for r in spark.sql(
        f"SELECT analytic_account_id FROM {CATALOG}.{SCHEMA}.account_analytic_account"
    ).collect():
        ana_pool.append(r.analytic_account_id)
except Exception:
    pass

merge_delta(spark.createDataFrame(ana_rows, schema=ana_schema),
            "account_analytic_account", "analytic_account_id")

# ═══════════════════════════════════════════════════════════════
# 4. BANK STATEMENTS (20 per run — append)
# ═══════════════════════════════════════════════════════════════
bs_id_start  = max_id("account_bank_statement",      "bank_statement_id")      + 1
bsl_id_start = max_id("account_bank_statement_line", "bank_statement_line_id") + 1

bs_rows  = []
bsl_rows = []
bs_id    = bs_id_start
bsl_id   = bsl_id_start

for _ in range(20):
    jid               = random.choice(BANK_JIDS)
    _, jname, _       = JOURNALS[jid]
    bdate             = rand_date()
    bal_s             = round(random.uniform(10_000, 500_000), 2)
    num_lines         = random.randint(2, 4)
    lines_data        = []
    net               = 0.0

    for seq in range(1, num_lines + 1):
        t_type   = random.choice(["credit"] * 60 + ["debit"] * 40)
        amount   = round(random.uniform(500, 20_000), 2) * (1 if t_type == "credit" else -1)
        rec      = random.random() < 0.55
        residual = 0.0 if rec else abs(amount)
        # FIX 2: use real partner
        pid = random.choice(customer_list + vendor_list)
        pname = customers_map.get(pid) or vendors_map.get(pid, "")
        lines_data.append((seq, t_type, amount, rec, residual, pid, pname))
        net += amount

    bal_e      = round(bal_s + net, 2)
    bal_e_real = round(bal_e * random.uniform(0.98, 1.02), 2)

    bs_rows.append(Row(
        bank_statement_id = bs_id,
        company_id        = COMPANY_ID,
        company_name      = COMPANY,
        currency_id       = 1,
        currency_name     = "EGP",
        journal_id        = jid,
        journal_name      = jname,
        name              = f"BNK/{bdate.year}/{bs_id:05d}",
        reference         = maybe(f"REF-{random.randint(1000,9999)}", 0.4),
        balance_start     = bal_s,
        balance_end       = bal_e,
        balance_end_real  = bal_e_real,
        is_complete       = random.random() < 0.6,
        date              = bdate,
        created_at        = bdate,
        updated_at        = bdate,
        dwh_loaded_at     = now,
        dwh_source_db     = SOURCE_DB,
    ))

    for seq, t_type, amount, rec, residual, pid, pname in lines_data:
        fcur_id = maybe(random.randint(2, 3), 0.7)
        bsl_rows.append(Row(
            bank_statement_line_id = bsl_id,
            statement_id           = bs_id,
            move_id                = None,
            move_name              = None,
            journal_id             = jid,
            journal_name           = jname,
            company_id             = COMPANY_ID,
            company_name           = COMPANY,
            partner_id             = pid,
            partner_name           = pname,
            currency_id            = 1,
            currency_name          = "EGP",
            foreign_currency_id    = fcur_id,
            foreign_currency_name  = CURRENCIES[fcur_id][0] if fcur_id else None,
            sequence               = seq,
            account_number         = f"EG{random.randint(10**18, 10**19-1)}",
            transaction_type       = t_type,
            payment_ref            = f"PAY-{random.randint(10000,99999)}",
            amount                 = amount,
            amount_currency        = round(amount * random.uniform(0.9, 1.1), 2),
            amount_residual        = residual,
            is_reconciled          = rec,
            created_at             = bdate,
            updated_at             = bdate,
            dwh_loaded_at          = now,
            dwh_source_db          = SOURCE_DB,
        ))
        bsl_id += 1
    bs_id += 1

# ═══════════════════════════════════════════════════════════════
# 5. ACCOUNT_MOVE → MOVE_LINES → PAYMENTS  (200 per run)
# ═══════════════════════════════════════════════════════════════
move_id_start  = max_id("account_move",      "account_move_id")      + 1
line_id_start  = max_id("account_move_line", "account_move_line_id") + 1
pay_id_start   = max_id("account_payment",   "account_payment_id")   + 1
anal_id_start2 = max_id("account_analytic_line", "analytic_line_id") + 1

move_rows  = []
ml_rows    = []
pay_rows   = []
anal_rows  = []

move_id  = move_id_start
line_id  = line_id_start
pay_id   = pay_id_start
anal_id  = anal_id_start2

# FIX 1: state pool
STATE_POOL = ["draft"]*10 + ["posted"]*80 + ["cancel"]*10

for _ in range(200):

    move_type = random.choice(["out_invoice", "in_invoice"])
    is_out    = move_type == "out_invoice"

    # FIX 1: realistic state distribution
    state     = random.choice(STATE_POOL)

    jid             = random.choice(SALE_JIDS if is_out else PURCHASE_JIDS)
    _, jname, _     = JOURNALS[jid]
    cur_id          = random.choice([1, 1, 1, 2, 3])
    cur_name, cur_r = CURRENCIES[cur_id]
    pt_id           = random.choice(list(PAYMENT_TERMS.keys()))
    date            = rand_date()
    due_days        = random.choice([15, 30, 45])
    date_due        = date + timedelta(days=due_days)
    del_date        = maybe(date + timedelta(days=random.randint(1, 10)), 0.4)

    # FIX 2: real partner from res_partner
    # FIX 5: link to real sale/purchase order and inherit partner+team
    so_id = po_id_ = None
    team_id = team_name = None

    if is_out and so_ids:
        so_id = random.choice(so_ids)
        so    = sale_orders[so_id]
        pid   = so["partner_id"] if so["partner_id"] in customers_map else random.choice(customer_list)
        team_id   = so["team_id"]    # FIX 11
        team_name = so["team_name"]
        origin    = f"SO-{so_id}"
        so_lines  = so["line_ids"]
    elif is_out:
        pid    = random.choice(customer_list)
        origin = maybe(f"SO-{random.randint(1,9999)}", 0.5)
        so_lines = []
    elif not is_out and po_ids:
        po_id_ = random.choice(po_ids)
        po     = purchase_orders[po_id_]
        pid    = po["partner_id"] if po["partner_id"] in vendors_map else random.choice(vendor_list)
        origin = f"PO-{po_id_}"
        po_lines = po["line_ids"]
    else:
        pid    = random.choice(vendor_list)
        origin = maybe(f"PO-{random.randint(1,9999)}", 0.5)
        po_lines = []

    pname = customers_map.get(pid) or vendors_map.get(pid, f"Partner_{pid}")

    # FIX 6: real user
    user_id   = random.choice(user_list)
    user_name = users_map[user_id]

    # FIX 9: real product & price-driven amounts
    prod_id    = random.choice(pid_list)
    prod       = products_map[prod_id]
    price_unit = prod["list_price"] if is_out else prod["standard_price"]
    qty        = float(random.randint(1, 10))
    discount   = float(random.choice([0, 0, 0, 5, 10, 15]))
    eff_price  = price_unit * (1 - discount / 100)
    base       = round(qty * eff_price, 2)
    tax_amt    = round(base * TAX_RATE, 2)
    total      = round(base + tax_amt, 2)

    # FIX 1: payment_state and residual depend on state
    if state == "cancel":
        # FIX 10: cancel = no residual, payment_state = not_paid
        pay_state = "not_paid"
        residual  = 0.0
    elif state == "draft":
        pay_state = "not_paid"
        residual  = total
    else:   # posted
        pay_state = random.choices(
            ["not_paid", "partial", "paid"], weights=[0.50, 0.30, 0.20]
        )[0]
        if pay_state == "paid":
            residual = 0.0
        elif pay_state == "partial":
            residual = round(total * random.uniform(0.20, 0.70), 2)
        else:
            residual = total

    sign       = 1.0 if is_out else -1.0
    rev_entry  = maybe(random.randint(max(1, move_id - 50), move_id - 1), 0.95) \
                 if move_id > move_id_start else None
    seq_pfx    = f"INV/{date.year}/" if is_out else f"BILL/{date.year}/"
    move_name  = f"{seq_pfx}{move_id:05d}"

    move_rows.append(Row(
        account_move_id                = move_id,
        journal_id                     = jid,
        journal_name                   = jname,
        company_id                     = COMPANY_ID,
        company_name                   = COMPANY,
        partner_id                     = pid,
        partner_name                   = pname,
        partner_shipping_id            = maybe(pid, 0.5),
        partner_shipping_name          = pname if random.random() > 0.5 else None,
        currency_id                    = cur_id,
        currency_name                  = cur_name,
        invoice_payment_term_id        = pt_id,
        invoice_payment_term_name      = PAYMENT_TERMS[pt_id],
        fiscal_position_id             = None,
        fiscal_position_name           = None,
        invoice_user_id                = user_id,
        invoice_user_name              = user_name,
        reversed_entry_id              = rev_entry,
        campaign_id                    = maybe(random.randint(1, 4), 0.7),
        campaign_name                  = maybe(f"Campaign_{random.randint(1,4)}", 0.7),
        team_id                        = team_id,
        team_name                      = team_name,
        sale_order_id                  = so_id,
        purchase_order_id              = po_id_,
        sequence_prefix                = seq_pfx,
        name                           = move_name,
        ref                            = maybe(f"REF-{random.randint(1000,9999)}", 0.5),
        state                          = state,          # FIX 1
        move_type                      = move_type,
        auto_post                      = "no",
        payment_reference              = maybe(f"PAY-{random.randint(10000,99999)}", 0.4),
        payment_state                  = pay_state,
        invoice_origin                 = origin,         # FIX 5
        invoice_partner_display_name   = pname,
        narration                      = maybe(f"Note {move_id}", 0.7),
        invoice_currency_rate          = cur_r,
        amount_untaxed                 = base,
        amount_tax                     = tax_amt,
        amount_total                   = total,
        amount_residual                = residual,       # FIX 10
        amount_untaxed_signed          = base * sign,
        amount_tax_signed              = tax_amt * sign,
        amount_total_signed            = total * sign,
        amount_residual_signed         = residual * sign,
        always_tax_exigible            = False,
        checked                        = False,
        posted_before                  = state == "posted",
        is_move_sent                   = state == "posted" and pay_state != "not_paid",
        date                           = date,
        invoice_date                   = date,
        invoice_date_due               = date_due,
        delivery_date                  = del_date,
        auto_post_until                = None,
        created_at                     = date,
        updated_at                     = date,
        dwh_loaded_at                  = now,
        dwh_source_db                  = SOURCE_DB,
    ))

    # ── MOVE LINES (3 per move — always balanced) ─────────────
    # FIX 9: price_unit from real product, qty drives total
    if is_out:
        line_defs = [
            ("asset_receivable",  total,    0.0,      prod_id, False, "payment_term"),
            ("income",            0.0,      base,     prod_id, False, "product"),
            ("liability_current", 0.0,      tax_amt,  None,    True,  "tax"),
        ]
    else:
        line_defs = [
            ("expense",           base,     0.0,      prod_id, False, "product"),
            ("asset_current",     tax_amt,  0.0,      None,    True,  "tax"),
            ("liability_payable", 0.0,      total,    None,    False, "payment_term"),
        ]

    for seq_n, (atype, debit, credit, lprod, is_tax, disp) in enumerate(line_defs, 1):
        balance = round(debit - credit, 2)
        tax_lid = 1 if is_tax else None

        # ADDED: link to source order line
        sol_id  = random.choice(so_lines)  if (is_out and so_lines  and lprod) else None
        pol_id_ = random.choice(po_lines)  if (not is_out and po_lines and lprod) else None

        ml_rows.append(Row(
            account_move_line_id    = line_id,
            move_id                 = move_id,
            account_id              = pick_acc(atype),
            account_name            = NAMES_MAP[atype][0],
            journal_id              = jid,
            journal_name            = jname,
            company_id              = COMPANY_ID,
            company_name            = COMPANY,
            currency_id             = cur_id,
            currency_name           = cur_name,
            partner_id              = pid,
            partner_name            = pname,
            product_id              = lprod,
            product_name            = prod["name"] if lprod else None,   # FIX 3
            product_uom_id          = 1 if lprod else None,
            product_uom_name        = "Unit(s)" if lprod else None,
            tax_line_id             = tax_lid,
            tax_line_name           = TAXES[tax_lid][0] if tax_lid else None,
            tax_group_id            = 1 if is_tax else None,
            tax_group_name          = "Tax Group 1" if is_tax else None,
            payment_id              = None,
            sale_order_line_id      = sol_id,    # ADDED
            purchase_order_line_id  = pol_id_,   # ADDED
            sequence                = seq_n,
            move_name               = move_name,
            parent_state            = state,
            ref                     = None,
            name                    = disp,
            display_type            = disp,
            analytic_distribution   = None,
            debit                   = round(debit,   2),
            credit                  = round(credit,  2),
            balance                 = balance,
            amount_currency         = round(balance * cur_r, 2),
            tax_base_amount         = base if is_tax else 0.0,
            amount_residual         = residual if disp == "payment_term" else 0.0,
            quantity                = qty if lprod else 0.0,
            price_unit              = price_unit if lprod else 0.0,  # FIX 9
            price_subtotal          = round(base,  2) if lprod else 0.0,
            price_total             = round(total, 2) if lprod else 0.0,
            discount                = discount if lprod else 0.0,
            is_storno               = False,
            reconciled              = (pay_state == "paid" and state == "posted"),
            is_downpayment          = False,
            date                    = date,
            invoice_date            = date,
            date_maturity           = date_due if disp == "payment_term" else None,
            created_at              = date,
            updated_at              = date,
            dwh_loaded_at           = now,
            dwh_source_db           = SOURCE_DB,
        ))

        # Analytic line — product lines only, posted invoices
        if lprod and ana_pool and state == "posted":
            ana_acc = random.choice(ana_pool)
            anal_rows.append(Row(
                analytic_line_id     = anal_id,
                account_id           = ana_acc,
                account_name         = f"ANA-{ana_acc:05d}",
                product_id           = lprod,
                product_name         = prod["name"],
                product_uom_id       = 1,
                product_uom_name     = "Unit(s)",
                partner_id           = pid,
                partner_name         = pname,
                company_id           = COMPANY_ID,
                company_name         = COMPANY,
                currency_id          = cur_id,
                currency_name        = cur_name,
                journal_id           = jid,
                journal_name         = jname,
                general_account_id   = pick_acc(atype),
                general_account_name = NAMES_MAP[atype][0],
                name                 = move_name,
                category             = "invoice" if is_out else "expense",
                code                 = f"ANA-{anal_id:06d}",
                ref                  = maybe(f"REF-{random.randint(1000,9999)}", 0.5),
                amount               = round(base * sign, 2),
                unit_amount          = qty,
                date                 = date,
                created_at           = date,
                updated_at           = date,
                dwh_loaded_at        = now,
                dwh_source_db        = SOURCE_DB,
            ))
            anal_id += 1

        line_id += 1

    # ── PAYMENT (only posted + paid/partial) ──────────────────
    if state == "posted" and pay_state in ("paid", "partial"):
        pay_amount = total if pay_state == "paid" else (total - residual)
        p_type     = "inbound" if is_out else "outbound"
        p_partner  = "customer" if is_out else "supplier"
        pm_id      = random.choice(PM_POOL)
        rec        = pay_state == "paid"
        bank_jid   = random.choice(BANK_JIDS)
        _, bank_jname, _ = JOURNALS[bank_jid]

        pay_rows.append(Row(
            account_payment_id              = pay_id,
            move_id                         = move_id,
            move_name                       = move_name,
            journal_id                      = bank_jid,
            journal_name                    = bank_jname,
            company_id                      = COMPANY_ID,
            company_name                    = COMPANY,
            partner_id                      = pid,
            partner_name                    = pname,
            currency_id                     = cur_id,
            currency_name                   = cur_name,
            source_payment_id               = None,
            payment_method_id               = pm_id,
            payment_method_name             = PAYMENT_METHODS[pm_id],
            name                            = f"PAY/{date.year}/{pay_id:05d}",
            state                           = "posted",
            payment_type                    = p_type,
            partner_type                    = p_partner,
            memo                            = maybe(f"Payment for {move_name}", 0.4),
            payment_reference               = maybe(f"REF-{random.randint(10000,99999)}", 0.5),
            amount                          = round(pay_amount, 2),
            amount_company_currency_signed  = round(pay_amount * sign * cur_r, 2),
            is_reconciled                   = rec,
            is_matched                      = rec,
            is_sent                         = True,
            date                            = date + timedelta(days=random.randint(0, 5)),
            created_at                      = date,
            updated_at                      = date,
            dwh_loaded_at                   = now,
            dwh_source_db                   = SOURCE_DB,
        ))
        pay_id += 1

    move_id += 1

# ═══════════════════════════════════════════════════════════════
# WRITE ALL TO DELTA
# ═══════════════════════════════════════════════════════════════
print("\n── Writing transactional tables ──")
append_delta(spark.createDataFrame(bs_rows,   schema=bs_schema),   "account_bank_statement")
append_delta(spark.createDataFrame(bsl_rows,  schema=bsl_schema),  "account_bank_statement_line")
append_delta(spark.createDataFrame(move_rows, schema=move_schema), "account_move")
append_delta(spark.createDataFrame(ml_rows,   schema=ml_schema),   "account_move_line")
append_delta(spark.createDataFrame(pay_rows,  schema=pay_schema),  "account_payment")
append_delta(spark.createDataFrame(anal_rows, schema=anal_schema), "account_analytic_line")

print(f"""
✅  Invoice simulation complete.
    account_move          : {len(move_rows)}   (draft={sum(1 for r in move_rows if r.state=='draft')} | posted={sum(1 for r in move_rows if r.state=='posted')} | cancel={sum(1 for r in move_rows if r.state=='cancel')})
    account_move_line     : {len(ml_rows)}
    account_payment       : {len(pay_rows)}
    account_analytic_line : {len(anal_rows)}
    bank_statement        : {len(bs_rows)}
    bank_statement_line   : {len(bsl_rows)}
""")